# Session 7, bank demo

The same three moves on the dataset your group project uses. **Watch in class**, then run it yourself afterwards.

45,211 customers of a Portuguese bank, contacted about a term deposit. About 11.7% said yes.

In [ ]:
# --- Load the data -------------------------------------------------
# The data loads from the course's public notebook repo, nothing to upload.
# No internet? Set DATA_URL = "" and upload the file when prompted (Colab only).
DATA_URL = "https://raw.githubusercontent.com/eoinlane/BUU33803-student-notebooks/main/session7/data/bank_customers.csv"
import pandas as pd
if DATA_URL:
    b = pd.read_csv(DATA_URL)
else:
    try:
        from google.colab import files
        print("Upload bank_customers.csv ...")
        b = pd.read_csv(next(iter(files.upload())))
    except Exception:
        b = pd.read_csv("data/bank_customers.csv")
b.head()

In [ ]:
print(b.shape)
print(b['y'].value_counts(normalize=True).round(3))

## 1 · Only what you know before you dial

This is the whole trick. Segment on information the bank has **before** it picks up the phone, otherwise the segments are useless for deciding who to call.

In [ ]:
PRE_CALL = ['age', 'balance', 'previous', 'job', 'marital',
            'education', 'housing', 'loan', 'poutcome']
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

X = pd.get_dummies(b[PRE_CALL], columns=['job', 'marital', 'education',
                                        'housing', 'loan', 'poutcome'])
Z = StandardScaler().fit_transform(X)
b['segment'] = KMeans(n_clusters=4, n_init=10, random_state=8).fit_predict(Z)

## 2 · Now look at what happened to each segment

**The outcome was never used to build the segments.** We clustered on customer attributes only, then looked up the subscribe rate afterwards.

In [ ]:
res = b.groupby('segment').apply(
    lambda g: pd.Series({
        'customers': len(g),
        'subscribed_%': round((g.y == 'yes').mean() * 100, 1),
        'avg_balance': round(g.balance.mean()),
        'has_housing_loan_%': round((g.housing == 'yes').mean() * 100),
        'has_personal_loan_%': round((g.loan == 'yes').mean() * 100),
        'prev_campaign_success_%': round((g.poutcome == 'success').mean() * 100, 1),
    }), include_groups=False)
res.sort_values('subscribed_%')

The spread is the point. Same script, same cost per call, and the best segment responds several times better than the worst.

**Clustering predicted nothing here.** It sorted the queue. That is a legitimate and very common commercial use of unsupervised learning, and it is different from the classification you did in Sessions 4 and 5.

## 3 · The trap, and why `duration` is not in the list

Run this to see what happens if you ignore the warning.

In [ ]:
bad = b.copy()
Zb = StandardScaler().fit_transform(bad[['age', 'balance', 'duration', 'campaign']])
bad['segment'] = KMeans(n_clusters=4, n_init=10, random_state=8).fit_predict(Zb)
print(bad.groupby('segment').apply(
    lambda g: pd.Series({'n': len(g),
                         'subscribed_%': round((g.y == 'yes').mean() * 100, 1),
                         'avg_call_seconds': round(g.duration.mean())}),
    include_groups=False).sort_values('subscribed_%'))

One segment leaps out. It is also **worthless**.

`duration` is how long the call lasted. You do not know it until the call is over. You cannot decide who to ring based on how long a conversation you have not had will run. The dataset's own documentation says to discard it for any realistic model.

**The general rule.** Before using a column, ask *when does its value become known?* If the answer is "after the thing I am trying to act on", it is leakage. It will flatter you in the notebook and fail in the business.

> This will come up in your group project. Check every column against that question before you build anything.